# 04 — Exploratory Data Analysis
## Nexora Supply Chain Analytics Platform

**Business objective:** uncover sales, profitability, customer, product, market, shipping, delivery, discount, and geographic patterns that can guide management decisions and Tableau dashboard design.

**Preferred input**
- `dataco_supply_chain_analytics_ready.csv`

**Fallback input**
- `dataco_supply_chain_cleaned.csv`

**Outputs**
- Professional visualizations
- KPI summary
- Ranked analytical tables
- Draft business insights
- `eda_summary_tables.xlsx`
- `eda_business_insights.csv`

## 1. Analytical scope

The EDA covers:

- Executive KPIs
- Sales and profit trends
- Market, region, category, and product performance
- Customer segments
- Shipping modes and late-delivery risk
- Discount and margin relationships
- Geographic performance
- Correlation and outlier review
- Management-priority segments

In [ ]:
import importlib.util
import subprocess
import sys

required = ["pandas", "numpy", "matplotlib", "plotly", "openpyxl"]
for package in required:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 150)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.titleweight"] = "bold"

## 2. Load the analytics-ready dataset

In [ ]:
FILE_CANDIDATES = [
    "/content/nexora_outputs/dataco_supply_chain_analytics_ready.csv",
    "/content/dataco_supply_chain_analytics_ready.csv",
    "/content/nexora_outputs/dataco_supply_chain_cleaned.csv",
    "/content/dataco_supply_chain_cleaned.csv",
    "/mnt/data/dataco_supply_chain_analytics_ready.csv",
]

def resolve_input_file(candidates):
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    try:
        from google.colab import files
        print("Upload the analytics-ready or cleaned CSV.")
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("No file was uploaded.")
        return next(iter(uploaded.keys()))
    except ImportError as exc:
        raise FileNotFoundError(
            "Dataset not found. Update FILE_CANDIDATES with a valid path."
        ) from exc

def read_csv_robust(path):
    for encoding in ("utf-8", "latin-1", "cp1252"):
        try:
            return pd.read_csv(path, encoding=encoding, low_memory=False)
        except UnicodeDecodeError:
            continue
    raise RuntimeError("Unable to detect CSV encoding.")

FILE_PATH = resolve_input_file(FILE_CANDIDATES)
df = read_csv_robust(FILE_PATH)

for col in [c for c in ["order_date", "shipping_date"] if c in df.columns]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

numeric_candidates = [
    "sales", "order_profit", "order_profit_per_order", "order_item_discount",
    "order_item_discount_rate", "discount_rate_pct", "profit_margin_pct",
    "days_for_shipping_real", "days_for_shipment_scheduled",
    "shipping_delay_days", "order_item_quantity", "latitude", "longitude",
    "is_late_delivery", "is_on_time_delivery"
]
for col in [c for c in numeric_candidates if c in df.columns]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Source: {FILE_PATH}")
print(f"Shape : {df.shape}")
display(df.head())

Upload the analytics-ready or cleaned CSV.


Saving dataco_supply_chain_analytics_ready.csv to dataco_supply_chain_analytics_ready.csv
Source: dataco_supply_chain_analytics_ready.csv
Shape : (180519, 85)


,payment_type,days_for_shipping,days_for_shipment,order_profit,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_id,customer_segment,customer_state,customer_zipcode,department_id,department_name,latitude,longitude,market,order_city,order_country,order_customer_id,order_date,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_state,order_status,order_zipcode,product_card_id,product_category_id,product_name,product_price,product_status,shipping_date,shipping_mode,is_late_delivery,profit_margin_pct,gross_sales_before_discount,profitability_status,order_year,order_quarter,order_month,order_month_name,order_week,order_day,order_day_name,order_date_key,shipping_date_key,order_quarter_number,order_year_month,is_weekend_order,shipping_year_month,discount_pct_calculated,discount_rate_pct,is_profitable_item,is_loss_item,sales_value_tier,margin_band,discount_band,order_total_sales,order_total_profit,order_total_quantity,order_item_count,order_profit_margin_pct,customer_order_count,customer_lifetime_sales,customer_average_item_sales,customer_lifetime_profit,customer_first_order_date,customer_last_order_date,customer_tenure_days,customer_frequency_segment,requires_management_attention,operational_risk_segment
0,CASH,2,4,88.79,239.98,Advance Shipping,0,43,Camping & Hiking,Hickory,Ee. Uu.,11599,Consumer,Nc,"28,601.00",7,Fan Shop,35.78,-81.36,LATAM,Mexico City,México,11599,2015-01-01 00:00:00,1,957,60.00,0.20,1,299.98,0.37,1,299.98,239.98,88.79,Central America,Distrito Federal,CLOSED,NaN,957,43,Diamondback Women's Serene Classic Comfort Bi,299.98,0,2015-01-03 00:00:00,Standard Class,0,29.60,359.98,Profitable,2015,Q1,1,January,1,1,Thursday,20150101,20150103,1,2015-01,0,2015-01,16.67,20.00,1,0,Premium,20–30%,10–20%,299.98,88.79,1,1,29.60,3,"1,244.94",207.49,266.99,2015-01-01 00:00:00,2017-05-24 13:08:00,874,Repeat,0,Healthy
1,PAYMENT,3,4,91.18,193.99,Advance Shipping,0,48,Water Sports,Chicago,Ee. Uu.,256,Consumer,Il,"60,625.00",7,Fan Shop,41.83,-87.98,LATAM,Dos Quebradas,Colombia,256,2015-01-01 00:21:00,2,1073,6.00,0.03,2,199.99,0.47,1,199.99,193.99,91.18,South America,Risaralda,PENDING_PAYMENT,NaN,1073,48,Pelican Sunstream 100 Kayak,199.99,0,2015-01-04 00:21:00,Standard Class,0,45.59,205.99,Profitable,2015,Q1,1,January,1,1,Thursday,20150101,20150104,1,2015-01,0,2015-01,2.91,3.00,1,0,High,30%+,0–5%,579.98,195.90,7,3,33.78,10,"4,169.48",189.52,"1,071.67",2015-01-01 00:21:00,2017-02-11 02:25:00,772,Loyal,0,Healthy
2,PAYMENT,3,4,68.25,227.50,Advance Shipping,0,24,Women's Apparel,Chicago,Ee. Uu.,256,Consumer,Il,"60,625.00",5,Golf,41.83,-87.98,LATAM,Dos Quebradas,Colombia,256,2015-01-01 00:21:00,2,502,22.50,0.09,3,50.00,0.30,5,250.00,227.50,68.25,South America,Risaralda,PENDING_PAYMENT,NaN,502,24,Nike Men's Dri-FIT Victory Golf Polo,50.00,0,2015-01-04 00:21:00,Standard Class,0,27.30,272.50,Profitable,2015,Q1,1,January,1,1,Thursday,20150101,20150104,1,2015-01,0,2015-01,8.26,9.00,1,0,High,20–30%,5–10%,579.98,195.90,7,3,33.78,10,"4,169.48",189.52,"1,071.67",2015-01-01 00:21:00,2017-02-11 02:25:00,772,Loyal,0,Healthy
3,PAYMENT,3,4,36.47,107.89,Advance Shipping,0,18,Men's Footwear,Chicago,Ee. Uu.,256,Consumer,Il,"60,625.00",4,Apparel,41.83,-87.98,LATAM,Dos Quebradas,Colombia,256,2015-01-01 00:21:00,2,403,22.10,0.17,4,129.99,0.34,1,129.99,107.89,36.47,South America,Risaralda,PENDING_PAYMENT,NaN,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.99,0,2015-01-04 00:21:00,Standard Class,0,28.06,152.09,Profitable,2015,Q1,1,January,1,1,Thursday,20150101,20150104,1,2015-01,0,2015-01,14.53,17.00,1,0,Medium,20–30%,10–20%,579.98,195.90,7,3,33.78,10,"4,169.48",189.52,"1,071.67",2015-01-01 00:21:00,2017-02-11 02:25:00,772,Loyal,0,Healthy
4,CASH,5,4,4.10,40.98,Late Delivery,1,40,Accessories,San Antonio,Ee. Uu.,8827,Home Office,T

## 3. Analytical helper functions

In [ ]:
profit_col = (
    "order_profit_per_order"
    if "order_profit_per_order" in df.columns
    else "order_profit" if "order_profit" in df.columns else None
)

customer_col = (
    "customer_id"
    if "customer_id" in df.columns
    else "order_customer_id" if "order_customer_id" in df.columns else None
)

def safe_divide(numerator, denominator):
    return numerator / denominator if denominator not in (0, None) else np.nan

def top_performance(group_col, n=10):
    agg = {"sales": "sum"} if "sales" in df.columns else {}
    if profit_col:
        agg[profit_col] = "sum"
    if "order_id" in df.columns:
        agg["order_id"] = "nunique"

    result = df.groupby(group_col, dropna=False).agg(agg).reset_index()
    rename_map = {"sales": "total_sales", profit_col: "total_profit", "order_id": "orders"}
    result = result.rename(columns={k: v for k, v in rename_map.items() if k in result.columns})
    if {"total_profit", "total_sales"}.issubset(result.columns):
        result["profit_margin_pct"] = np.where(
            result["total_sales"].ne(0),
            result["total_profit"] / result["total_sales"] * 100,
            np.nan,
        )
    return result.sort_values("total_sales", ascending=False).head(n)

def plot_bar(data, x, y, title, orientation="v"):
    if orientation == "h":
        fig = px.bar(data, x=y, y=x, orientation="h", title=title, text_auto=".3s")
    else:
        fig = px.bar(data, x=x, y=y, title=title, text_auto=".3s")
    fig.update_layout(template="plotly_white")
    fig.show()

## 4. Executive KPI overview

These KPIs provide the headline performance view for the executive dashboard.

In [ ]:
total_sales = df["sales"].sum() if "sales" in df.columns else np.nan
total_profit = df[profit_col].sum() if profit_col else np.nan
total_orders = df["order_id"].nunique() if "order_id" in df.columns else np.nan
total_customers = df[customer_col].nunique() if customer_col else np.nan
total_items = len(df)
avg_order_value = safe_divide(total_sales, total_orders)

if "is_late_delivery" in df.columns:
    late_delivery_rate = df["is_late_delivery"].mean() * 100
elif "late_delivery_risk" in df.columns:
    late_delivery_rate = pd.to_numeric(df["late_delivery_risk"], errors="coerce").mean() * 100
else:
    late_delivery_rate = np.nan

overall_margin = safe_divide(total_profit, total_sales) * 100

kpi_table = pd.DataFrame({
    "KPI": [
        "Total Sales", "Total Profit", "Profit Margin",
        "Distinct Orders", "Distinct Customers",
        "Order Item Rows", "Average Order Value", "Late Delivery Rate"
    ],
    "Value": [
        total_sales, total_profit, overall_margin,
        total_orders, total_customers, total_items,
        avg_order_value, late_delivery_rate
    ],
})
display(kpi_table)

,KPI,Value
0,Total Sales,"36,784,734.31"
1,Total Profit,"3,966,902.97"
2,Profit Margin,10.78
3,Distinct Orders,"65,752.00"
4,Distinct Customers,"20,652.00"
5,Order Item Rows,"180,519.00"
6,Average Order Value,559.45
7,Late Delivery Rate,54.83


## 5. Monthly sales and profit trend

In [ ]:
if "order_date" in df.columns and "sales" in df.columns:
    monthly = (
        df.dropna(subset=["order_date"])
          .assign(order_month=df["order_date"].dt.to_period("M").astype(str))
          .groupby("order_month", as_index=False)
          .agg(
              total_sales=("sales", "sum"),
              total_profit=(profit_col, "sum") if profit_col else ("sales", "size"),
              orders=("order_id", "nunique") if "order_id" in df.columns else ("sales", "size"),
          )
    )

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=monthly["order_month"], y=monthly["total_sales"],
        mode="lines+markers", name="Sales"
    ))
    if profit_col:
        fig.add_trace(go.Scatter(
            x=monthly["order_month"], y=monthly["total_profit"],
            mode="lines+markers", name="Profit"
        ))
    fig.update_layout(
        title="Monthly Sales and Profit Trend",
        xaxis_title="Month",
        yaxis_title="Value",
        template="plotly_white",
        hovermode="x unified",
    )
    fig.show()
    display(monthly.tail(12))

,order_month,total_sales,total_profit,orders
25,2017-02,"992,534.88","115,111.07",1614
26,2017-03,"1,048,004.76","118,688.90",1782
27,2017-04,"1,038,321.60","115,961.06",1739
28,2017-05,"1,105,485.30","115,014.64",1763
29,2017-06,"1,032,086.47","110,399.29",1676
30,2017-07,"1,104,373.34","113,026.70",1776
31,2017-08,"1,109,337.15","131,501.16",1768
32,2017-09,"1,143,775.09","122,462.39",1723
33,2017-10,"1,073,994.16","113,447.17",2101
34,2017-11,"626,914.37","67,791.25",2055


## 6. Market and regional performance

In [ ]:
market_tables = {}

for dimension in ["market", "order_region", "order_country"]:
    if dimension in df.columns:
        table = top_performance(dimension, n=15)
        market_tables[dimension] = table
        plot_bar(
            table.sort_values("total_sales"),
            dimension,
            "total_sales",
            f"Sales Performance by {dimension.replace('_', ' ').title()}",
            orientation="h",
        )
        display(table)

,market,total_sales,total_profit,orders,profit_margin_pct
1,Europe,"10,872,396.60","1,169,442.96",18561,10.76
2,LATAM,"10,277,612.64","1,123,321.61",17181,10.93
3,Pacific Asia,"8,273,743.58","857,753.44",17577,10.37
4,USCA,"5,066,528.61","564,313.78",8579,11.14
0,Africa,"2,294,452.88","252,071.18",3854,10.99


,order_region,total_sales,total_profit,orders,profit_margin_pct
22,Western Europe,"5,894,380.66","625,446.08",10010,10.61
3,Central America,"5,665,711.99","616,341.57",9396,10.88
12,South America,"2,960,881.35","335,154.40",4979,11.32
10,Northern Europe,"2,155,830.61","233,450.60",3716,10.83
17,Southern Europe,"2,047,918.78","230,829.23",3543,11.27
11,Oceania,"2,016,654.16","201,478.02",4362,9.99
15,Southeast Asia,"1,932,495.53","211,342.82",4356,10.94
1,Caribbean,"1,651,019.30","171,825.64",2806,10.41
21,West of USA,"1,571,415.93","164,940.66",2667,10.50
13,South Asia,"1,553,680.89","165,703.90",3335,10.67


,order_country,total_sales,total_profit,orders,profit_margin_pct
48,Estados Unidos,"4,879,667.57","540,413.07",8270,11.07
53,Francia,"2,879,942.31","327,828.58",4866,11.38
102,México,"2,633,195.24","303,278.37",4395,11.52
2,Alemania,"2,074,171.78","194,827.08",3518,9.39
8,Australia,"1,694,621.64","170,041.58",3798,10.03
120,Reino Unido,"1,612,094.82","180,942.88",2785,11.22
20,Brasil,"1,594,319.92","186,713.64",2650,11.71
31,China,"1,172,902.09","122,190.92",2616,10.42
75,Italia,"1,072,181.65","121,545.47",1880,11.34
69,India,"962,396.68","99,746.82",2152,10.36


## 7. Category and product analysis

In [ ]:
product_tables = {}

for dimension, n in [("category_name", 15), ("product_name", 15), ("department_name", 15)]:
    if dimension in df.columns:
        table = top_performance(dimension, n=n)
        product_tables[dimension] = table
        plot_bar(
            table.sort_values("total_sales"),
            dimension,
            "total_sales",
            f"Top {n} {dimension.replace('_', ' ').title()} by Sales",
            orientation="h",
        )
        display(table)

,category_name,total_sales,total_profit,orders,profit_margin_pct
18,Fishing,"6,929,653.50","756,220.76",15164,10.91
12,Cleats,"4,431,942.66","494,636.92",20386,11.16
9,Camping & Hiking,"4,118,425.42","427,455.57",12299,10.38
10,Cardio Equipment,"3,694,843.20","383,011.10",11355,10.37
47,Women's Apparel,"3,147,800.00","350,421.03",17869,11.13
46,Water Sports,"3,113,844.60","325,146.96",13758,10.44
34,Men's Footwear,"2,891,757.54","311,902.82",18783,10.79
30,Indoor/Outdoor Games,"2,888,993.94","318,451.43",16623,11.02
38,Shop By Sport,"1,309,522.02","129,813.96",10136,9.91
13,Computers,"663,000.00","69,656.81",442,10.51


,product_name,total_sales,total_profit,orders,profit_margin_pct
24,Field & Stream Sportsman 16 Gun Fire Safe,"6,929,653.50","756,220.76",15164,10.91
71,Perfect Fitness Perfect Rip Deck,"4,421,143.02","493,828.30",20359,11.17
21,Diamondback Women's Serene Classic Comfort Bi,"4,118,425.42","427,455.57",12299,10.38
61,Nike Men's Free 5.0+ Running Shoe,"3,667,633.20","379,915.82",11092,10.36
59,Nike Men's Dri-FIT Victory Golf Polo,"3,147,800.00","350,421.03",17869,11.13
70,Pelican Sunstream 100 Kayak,"3,099,845.00","324,076.37",13727,10.45
56,Nike Men's CJ Elite 2 TD Football Cleat,"2,891,757.54","311,902.82",18783,10.79
67,O'Brien Men's Neoprene Life Vest,"2,888,993.94","318,451.43",16623,11.02
102,Under Armour Girls' Toddler Spine Surge Runni,"1,269,082.65","126,278.51",9825,9.95
18,Dell Laptop,"663,000.00","69,656.81",442,10.51


,department_name,total_sales,total_profit,orders,profit_margin_pct
3,Fan Shop,"17,113,870.54","1,834,155.43",41303,10.72
0,Apparel,"7,976,255.08","881,882.93",35190,11.06
6,Golf,"4,609,028.22","497,523.56",25889,10.79
5,Footwear,"4,006,498.77","410,222.50",13009,10.24
8,Outdoors,"1,253,351.44","145,251.46",9066,11.59
10,Technology,"1,039,598.96","113,170.01",1465,10.89
4,Fitness,"397,050.89","46,538.06",2437,11.72
2,Discs Shop,"228,887.73","24,193.12",2026,10.57
7,Health and Beauty,"106,080.48","9,493.63",362,8.95
9,Pet Shop,"41,524.80","3,589.26",492,8.64


## 8. Profitability and loss concentration

High sales do not always mean high profit. This section identifies segments with negative profit or weak margins.

In [ ]:
loss_tables = {}

if profit_col:
    dimension_candidates = [
        "category_name", "product_name", "market",
        "order_region", "shipping_mode", "customer_segment"
    ]

    for dimension in [c for c in dimension_candidates if c in df.columns]:
        table = (
            df.groupby(dimension, dropna=False)
              .agg(
                  total_sales=("sales", "sum"),
                  total_profit=(profit_col, "sum"),
                  item_rows=(profit_col, "size"),
                  loss_items=(profit_col, lambda s: int((s < 0).sum())),
              )
              .reset_index()
        )
        table["profit_margin_pct"] = np.where(
            table["total_sales"].ne(0),
            table["total_profit"] / table["total_sales"] * 100,
            np.nan,
        )
        table["loss_item_rate_pct"] = table["loss_items"] / table["item_rows"] * 100
        loss_tables[dimension] = table.sort_values("total_profit")

    primary_loss_dimension = (
        "category_name" if "category_name" in loss_tables
        else next(iter(loss_tables), None)
    )
    if primary_loss_dimension:
        worst = loss_tables[primary_loss_dimension].head(15)
        plot_bar(
            worst,
            primary_loss_dimension,
            "total_profit",
            f"Lowest-Profit {primary_loss_dimension.replace('_', ' ').title()}",
            orientation="h",
        )
        display(worst)

,category_name,total_sales,total_profit,item_rows,loss_items,profit_margin_pct,loss_item_rate_pct
41,Strength Training,"54,895.53",332.31,111,18,0.61,16.22
7,CDs,"3,059.59",383.85,271,36,12.55,13.28
1,As Seen on TV!,"20,597.94",714.43,68,14,3.47,20.59
5,Books,"12,587.40",883.01,405,78,7.02,19.26
43,Toys,"6,104.66",900.71,529,82,14.75,15.50
2,Baby,"12,229.56","1,525.03",207,34,12.47,16.43
23,Golf Bags & Carts,"10,369.39","1,810.07",61,6,17.46,9.84
4,Basketball,"27,099.33","1,845.67",67,11,6.81,16.42
33,Men's Clothing,"43,856.80","2,006.04",208,50,4.57,24.04
45,Video Games,"33,310.50","2,717.52",838,181,8.16,21.60


## 9. Customer-segment analysis

In [ ]:
customer_tables = {}

for dimension in ["customer_segment", "customer_frequency_segment"]:
    if dimension in df.columns:
        agg_spec = {"total_sales": ("sales", "sum")}
        if profit_col:
            agg_spec["total_profit"] = (profit_col, "sum")
        if customer_col:
            agg_spec["customers"] = (customer_col, "nunique")
        if "order_id" in df.columns:
            agg_spec["orders"] = ("order_id", "nunique")

        table = df.groupby(dimension, dropna=False).agg(**agg_spec).reset_index()

        if {"total_sales", "customers"}.issubset(table.columns):
            table["sales_per_customer"] = np.where(
                table["customers"].ne(0),
                table["total_sales"] / table["customers"],
                np.nan,
            )
        if {"total_profit", "total_sales"}.issubset(table.columns):
            table["profit_margin_pct"] = np.where(
                table["total_sales"].ne(0),
                table["total_profit"] / table["total_sales"] * 100,
                np.nan,
            )

        customer_tables[dimension] = table.sort_values("total_sales", ascending=False)
        plot_bar(
            customer_tables[dimension],
            dimension,
            "total_sales",
            f"Sales by {dimension.replace('_', ' ').title()}",
        )
        display(customer_tables[dimension])

,customer_segment,total_sales,total_profit,customers,orders,sales_per_customer,profit_margin_pct
0,Consumer,"19,095,789.79","2,073,487.67",10695,34119,"1,785.49",10.86
1,Corporate,"11,168,406.63","1,202,574.96",6239,19856,"1,790.10",10.77
2,Home Office,"6,520,537.89","690,840.34",3718,11777,"1,753.78",10.59


,customer_frequency_segment,total_sales,total_profit,customers,orders,sales_per_customer,profit_margin_pct
0,Loyal,"15,890,819.56","1,712,985.14",3805,26673,"4,176.30",10.78
3,Repeat,"15,655,658.41","1,665,591.33",6484,26120,"2,414.51",10.64
2,One-Time,"2,794,632.00","307,949.76",8884,8884,314.57,11.02
1,Occasional,"1,636,261.35","191,242.64",1363,2726,"1,200.49",11.69
4,VIP,"807,362.99","89,134.10",116,1349,"6,960.03",11.04


## 10. Shipping-mode and delivery-performance analysis

In [ ]:
delivery_tables = {}

if "shipping_mode" in df.columns:
    agg_spec = {
        "item_rows": ("shipping_mode", "size"),
        "total_sales": ("sales", "sum"),
    }
    if profit_col:
        agg_spec["total_profit"] = (profit_col, "sum")
    if "days_for_shipping_real" in df.columns:
        agg_spec["avg_actual_shipping_days"] = ("days_for_shipping_real", "mean")
    if "shipping_delay_days" in df.columns:
        agg_spec["avg_delay_days"] = ("shipping_delay_days", "mean")
    if "is_late_delivery" in df.columns:
        agg_spec["late_delivery_rate_pct"] = ("is_late_delivery", lambda s: s.mean() * 100)

    shipping_summary = (
        df.groupby("shipping_mode", dropna=False)
          .agg(**agg_spec)
          .reset_index()
          .sort_values("late_delivery_rate_pct" if "late_delivery_rate_pct" in agg_spec else "total_sales", ascending=False)
    )
    delivery_tables["shipping_mode"] = shipping_summary
    display(shipping_summary)

    metric = "late_delivery_rate_pct" if "late_delivery_rate_pct" in shipping_summary.columns else "total_sales"
    plot_bar(shipping_summary, "shipping_mode", metric, f"{metric.replace('_', ' ').title()} by Shipping Mode")

if "delivery_status" in df.columns:
    delivery_status = (
        df.groupby("delivery_status", dropna=False)
          .agg(
              item_rows=("delivery_status", "size"),
              total_sales=("sales", "sum"),
          )
          .reset_index()
          .sort_values("item_rows", ascending=False)
    )
    delivery_tables["delivery_status"] = delivery_status
    fig = px.pie(
        delivery_status,
        names="delivery_status",
        values="item_rows",
        title="Distribution of Delivery Status",
        hole=0.45,
    )
    fig.update_layout(template="plotly_white")
    fig.show()
    display(delivery_status)

,shipping_mode,item_rows,total_sales,total_profit,late_delivery_rate_pct
0,First Class,27814,"5,674,369.65","643,121.92",95.32
2,Second Class,35216,"7,145,444.68","750,308.17",76.63
1,Same Day,9737,"1,942,528.52","203,018.43",45.74
3,Standard Class,107752,"22,022,391.46","2,370,454.45",38.07


,delivery_status,item_rows,total_sales
1,Late Delivery,98977,"20,126,394.88"
0,Advance Shipping,41592,"8,518,007.73"
3,Shipping On Time,32196,"6,570,026.37"
2,Shipping Canceled,7754,"1,570,305.33"


## 11. Late-delivery heatmap by market and shipping mode

In [ ]:
if {"market", "shipping_mode", "is_late_delivery"}.issubset(df.columns):
    late_pivot = pd.pivot_table(
        df,
        index="market",
        columns="shipping_mode",
        values="is_late_delivery",
        aggfunc="mean",
    ) * 100

    fig = px.imshow(
        late_pivot.round(2),
        text_auto=True,
        aspect="auto",
        title="Late Delivery Rate (%) by Market and Shipping Mode",
        labels={"color": "Late Delivery Rate (%)"},
    )
    fig.update_layout(template="plotly_white")
    fig.show()
    display(late_pivot.round(2))

shipping_mode,First Class,Same Day,Second Class,Standard Class
market,,,,
Africa,96.99,44.01,77.40,38.21
Europe,95.82,46.43,77.03,38.01
LATAM,94.74,48.04,75.56,37.89
Pacific Asia,95.08,43.53,77.24,38.47
USCA,95.16,44.42,76.65,37.86


## 12. Discount, sales, and margin relationship

A sample is used for interactive scatter plots to maintain responsive performance in Colab.

In [ ]:
scatter_columns = [
    c for c in [
        "sales", profit_col, "profit_margin_pct",
        "discount_rate_pct", "order_item_discount_rate",
        "order_item_discount", "category_name", "market"
    ] if c and c in df.columns
]

sample_n = min(20_000, len(df))
scatter_df = df[scatter_columns].dropna().sample(
    n=min(sample_n, len(df[scatter_columns].dropna())),
    random_state=42,
) if len(df[scatter_columns].dropna()) else pd.DataFrame()

discount_x = (
    "discount_rate_pct"
    if "discount_rate_pct" in scatter_df.columns
    else "order_item_discount_rate" if "order_item_discount_rate" in scatter_df.columns
    else "order_item_discount" if "order_item_discount" in scatter_df.columns else None
)

scatter_y = "profit_margin_pct" if "profit_margin_pct" in scatter_df.columns else profit_col

if not scatter_df.empty and discount_x and scatter_y:
    fig = px.scatter(
        scatter_df,
        x=discount_x,
        y=scatter_y,
        color="market" if "market" in scatter_df.columns else None,
        hover_data=["category_name"] if "category_name" in scatter_df.columns else None,
        opacity=0.45,
        title="Discount and Profitability Relationship",
    )
    fig.update_layout(template="plotly_white")
    fig.show()

## 13. Correlation analysis

In [ ]:
correlation_candidates = [
    "sales", profit_col, "profit_margin_pct",
    "order_item_discount", "discount_rate_pct",
    "order_item_product_price", "order_item_quantity",
    "days_for_shipping_real", "days_for_shipment_scheduled",
    "shipping_delay_days", "is_late_delivery"
]
correlation_cols = [c for c in correlation_candidates if c and c in df.columns]

if len(correlation_cols) >= 2:
    corr = df[correlation_cols].corr(numeric_only=True)
    fig = px.imshow(
        corr.round(2),
        text_auto=True,
        aspect="auto",
        title="Correlation Matrix of Core Business Metrics",
        zmin=-1,
        zmax=1,
    )
    fig.update_layout(template="plotly_white")
    fig.show()
    display(corr.round(3))

,sales,order_profit_per_order,profit_margin_pct,order_item_discount,discount_rate_pct,order_item_product_price,order_item_quantity,is_late_delivery
sales,1.00,0.13,-0.00,0.62,0.00,0.79,0.11,-0.00
order_profit_per_order,0.13,1.00,0.83,0.07,-0.02,0.10,0.02,-0.00
profit_margin_pct,-0.00,0.83,1.00,-0.02,-0.02,-0.00,0.00,-0.00
order_item_discount,0.62,0.07,-0.02,1.00,0.66,0.49,0.07,-0.00
discount_rate_pct,0.00,-0.02,-0.02,0.66,1.00,0.00,-0.00,0.00
order_item_product_price,0.79,0.10,-0.00,0.49,0.00,1.00,-0.48,-0.00
order_item_quantity,0.11,0.02,0.00,0.07,-0.00,-0.48,1.00,-0.00
is_late_delivery,-0.00,-0.00,-0.00,-0.00,0.00,-0.00,-0.00,1.00


## 14. Outlier review using the IQR method

Outliers are flagged for investigation, not automatically deleted. Extreme orders can be valid and commercially important.

In [ ]:
outlier_columns = [
    c for c in [
        "sales", profit_col, "order_item_discount",
        "profit_margin_pct", "days_for_shipping_real",
        "shipping_delay_days"
    ] if c and c in df.columns
]

outlier_summary_rows = []
for col in outlier_columns:
    series = df[col].dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = int(((df[col] < lower) | (df[col] > upper)).sum())

    outlier_summary_rows.append({
        "column": col,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": count,
        "outlier_pct": count / len(df) * 100,
    })

outlier_summary = pd.DataFrame(outlier_summary_rows)
display(outlier_summary)

,column,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_pct
0,sales,119.98,299.95,179.97,-149.97,569.90,488,0.27
1,order_profit_per_order,7.00,64.80,57.80,-79.70,151.50,18942,10.49
2,order_item_discount,5.40,29.99,24.59,-31.48,66.87,7537,4.18
3,profit_margin_pct,6.22,33.60,27.38,-34.84,74.67,16829,9.32


## 15. Geographic analysis

In [ ]:
geographic_tables = {}

for dimension in ["order_country", "order_state", "order_city"]:
    if dimension in df.columns:
        table = top_performance(dimension, n=15)
        geographic_tables[dimension] = table
        plot_bar(
            table.sort_values("total_sales"),
            dimension,
            "total_sales",
            f"Top Locations by Sales — {dimension.replace('_', ' ').title()}",
            orientation="h",
        )

if {"longitude", "latitude", "sales"}.issubset(df.columns):
    geo = (
        df.dropna(subset=["longitude", "latitude"])
          .groupby(["longitude", "latitude"], as_index=False)
          .agg(total_sales=("sales", "sum"), item_rows=("sales", "size"))
    )
    if len(geo):
        map_sample = geo.nlargest(min(5_000, len(geo)), "total_sales")
        fig = px.scatter_geo(
            map_sample,
            lon="longitude",
            lat="latitude",
            size="total_sales",
            hover_data=["item_rows"],
            title="Geographic Sales Concentration",
            projection="natural earth",
        )
        fig.update_layout(template="plotly_white")
        fig.show()

## 16. Management-priority analysis

In [ ]:
attention_tables = {}

if "operational_risk_segment" in df.columns:
    agg_spec = {
        "item_rows": ("operational_risk_segment", "size"),
        "total_sales": ("sales", "sum"),
    }
    if profit_col:
        agg_spec["total_profit"] = (profit_col, "sum")

    risk_summary = (
        df.groupby("operational_risk_segment", dropna=False)
          .agg(**agg_spec)
          .reset_index()
          .sort_values("item_rows", ascending=False)
    )
    attention_tables["operational_risk_segment"] = risk_summary
    plot_bar(
        risk_summary,
        "operational_risk_segment",
        "item_rows",
        "Order Items by Operational Risk Segment",
    )
    display(risk_summary)

if {"requires_management_attention", "category_name"}.issubset(df.columns):
    priority_category = (
        df.groupby("category_name", dropna=False)
          .agg(
              item_rows=("category_name", "size"),
              attention_items=("requires_management_attention", "sum"),
              total_sales=("sales", "sum"),
              total_profit=(profit_col, "sum") if profit_col else ("sales", "size"),
          )
          .reset_index()
    )
    priority_category["attention_rate_pct"] = (
        priority_category["attention_items"] / priority_category["item_rows"] * 100
    )
    priority_category = priority_category.sort_values(
        ["attention_rate_pct", "total_sales"], ascending=[False, False]
    )
    attention_tables["priority_category"] = priority_category
    display(priority_category.head(20))

,operational_risk_segment,item_rows,total_sales,total_profit
1,Late Only,80424,"16,345,871.24","4,293,678.29"
0,Healthy,66311,"13,568,691.16","3,556,772.03"
2,Late and Loss,18553,"3,780,523.64","-2,153,626.61"
3,Loss Only,15231,"3,089,648.27","-1,729,920.74"


,category_name,item_rows,attention_items,total_sales,total_profit,attention_rate_pct
23,Golf Bags & Carts,61,43,"10,369.39","1,810.07",70.49
32,Lacrosse,343,236,"39,464.79","4,364.29",68.80
37,Pet Supplies,492,336,"41,524.80","3,589.26",68.29
19,Fitness Accessories,309,210,"35,601.44","5,258.39",67.96
27,Health and Beauty,362,243,"106,080.48","9,493.63",67.13
42,Tennis & Racquet,328,219,"44,585.09","5,747.98",66.77
15,Crafts,484,322,"223,356.32","25,531.17",66.53
26,Golf Shoes,524,347,"107,998.00","12,406.07",66.22
8,Cameras,592,390,"267,607.68","30,289.80",65.88
36,Music,434,282,"113,122.10","14,436.32",64.98


## 17. Automatically generated draft insights

These statements are calculated from the dataset and should be reviewed before being published as final management conclusions.

In [ ]:
insight_rows = []

def add_insight(area, finding, implication):
    insight_rows.append({
        "analysis_area": area,
        "finding": finding,
        "business_implication": implication,
    })

add_insight(
    "Executive",
    f"Total sales are {total_sales:,.2f} with total profit of {total_profit:,.2f}."
    if profit_col else f"Total sales are {total_sales:,.2f}.",
    "Use this as the baseline for executive performance tracking.",
)

if pd.notna(late_delivery_rate):
    add_insight(
        "Delivery",
        f"The overall late-delivery rate is {late_delivery_rate:,.2f}%.",
        "Prioritize markets and shipping modes with rates above this baseline.",
    )

if "market" in market_tables and len(market_tables["market"]):
    top = market_tables["market"].iloc[0]
    add_insight(
        "Market",
        f"{top['market']} is the largest market by sales at {top['total_sales']:,.2f}.",
        "Protect service quality and product availability in this core market.",
    )

if "category_name" in product_tables and len(product_tables["category_name"]):
    top = product_tables["category_name"].iloc[0]
    add_insight(
        "Category",
        f"{top['category_name']} is the highest-sales category at {top['total_sales']:,.2f}.",
        "Use the category as a benchmark while checking margin and delivery performance.",
    )

if profit_col and "category_name" in loss_tables and len(loss_tables["category_name"]):
    worst = loss_tables["category_name"].iloc[0]
    add_insight(
        "Profitability",
        f"{worst['category_name']} has the lowest category profit at {worst['total_profit']:,.2f}.",
        "Review pricing, discounting, product mix, and fulfillment cost drivers.",
    )

insights = pd.DataFrame(insight_rows)
display(insights)

,analysis_area,finding,business_implication
0,Executive,"Total sales are 36,784,734.31 with total profi...",Use this as the baseline for executive perform...
1,Delivery,The overall late-delivery rate is 54.83%.,Prioritize markets and shipping modes with rat...
2,Market,"Europe is the largest market by sales at 10,87...",Protect service quality and product availabili...
3,Category,"Fishing is the highest-sales category at 6,929...",Use the category as a benchmark while checking...
4,Profitability,Strength Training has the lowest category prof...,"Review pricing, discounting, product mix, and ..."


## 18. Export EDA tables and insight drafts

In [ ]:
OUTPUT_DIR = Path("/content/nexora_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXCEL_OUTPUT = OUTPUT_DIR / "eda_summary_tables.xlsx"
INSIGHTS_OUTPUT = OUTPUT_DIR / "eda_business_insights.csv"
OUTLIER_OUTPUT = OUTPUT_DIR / "eda_outlier_summary.csv"

with pd.ExcelWriter(EXCEL_OUTPUT, engine="openpyxl") as writer:
    kpi_table.to_excel(writer, sheet_name="executive_kpis", index=False)

    if "monthly" in globals():
        monthly.to_excel(writer, sheet_name="monthly_trend", index=False)

    for name, table in market_tables.items():
        table.to_excel(writer, sheet_name=f"market_{name}"[:31], index=False)

    for name, table in product_tables.items():
        table.to_excel(writer, sheet_name=f"product_{name}"[:31], index=False)

    for name, table in customer_tables.items():
        table.to_excel(writer, sheet_name=f"customer_{name}"[:31], index=False)

    for name, table in delivery_tables.items():
        table.to_excel(writer, sheet_name=f"delivery_{name}"[:31], index=False)

    for name, table in geographic_tables.items():
        table.to_excel(writer, sheet_name=f"geo_{name}"[:31], index=False)

    for name, table in attention_tables.items():
        table.to_excel(writer, sheet_name=f"risk_{name}"[:31], index=False)

    outlier_summary.to_excel(writer, sheet_name="outlier_summary", index=False)

insights.to_csv(INSIGHTS_OUTPUT, index=False, encoding="utf-8")
outlier_summary.to_csv(OUTLIER_OUTPUT, index=False, encoding="utf-8")

print(f"EDA workbook      : {EXCEL_OUTPUT}")
print(f"Insight draft     : {INSIGHTS_OUTPUT}")
print(f"Outlier summary   : {OUTLIER_OUTPUT}")

EDA workbook      : /content/nexora_outputs/eda_summary_tables.xlsx
Insight draft     : /content/nexora_outputs/eda_business_insights.csv
Outlier summary   : /content/nexora_outputs/eda_outlier_summary.csv


## 19. EDA completion checklist

- [ ] Executive KPIs were successfully calculated.
- [ ] Monthly trends were reviewed for seasonality or anomalies.
- [ ] Top and bottom markets, regions, categories, and products were identified.
- [ ] Profit and sales were evaluated together.
- [ ] Customer segments were compared.
- [ ] Shipping modes were evaluated using delivery metrics.
- [ ] Discount and profitability relationships were reviewed.
- [ ] Correlation does not automatically imply causation.
- [ ] Outliers were investigated rather than automatically removed.
- [ ] EDA summary tables and draft insights were exported.
- [ ] Final business conclusions will be supported by SQL and Tableau analysis.